<a href="https://colab.research.google.com/github/PavanRishi69/dataviz-exercises-pavan-rishi-gillella/blob/main/lecture06_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 6 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week06/lecture06_exercise.ipynb`

**Rules:**
1. Use `px` first, then customise with `update_traces` / `update_layout`
2. Colour encodes a meaningful category — not decoration
3. Insight title names the specific finding
4. Consider: would a bar chart be clearer? If yes, use the bar chart

---


In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('/content/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         60  7168.0   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [2]:
# Task 1
# Filter only fossil fuels
fossil_df = df[df['Source_Type'] == 'Fossil']

# Find most fossil-dependent country
top_country = (
    fossil_df.groupby('Country')['TWh']
    .sum()
    .idxmax()
)

# Create treemap
fig = px.treemap(
    fossil_df,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_sequence=px.colors.qualitative.Safe   # CVD-safe palette
)

# Show TWh values only (no percentages)
fig.update_traces(
    textinfo='label+value',
    root_color='lightgrey'   # Grey parent/root nodes
)

# Title with insight
fig.update_layout(
    title=f'Fossil Fuel Dependency by Country — Highest dependency: {top_country}'
)

fig.show()


### Insights for Task 1 (Treemap — Fossil Fuel Dependency)

1. **Countries with larger treemap sections have higher dependence on fossil fuels, indicating stronger reliance on non-renewable energy sources.**

2. **Coal, oil, and natural gas contributions vary across countries, showing differences in energy production strategies between regions.**

3. **The country highlighted in the title as having the highest fossil fuel generation appears to be the most dependent on fossil-based energy, which may increase environmental impact.**

4. **Regional patterns in the treemap suggest some areas are more reliant on fossil fuels than others, reflecting uneven progress toward cleaner energy adoption.**

These insights match the requirements of interpreting **country dependence + fossil source contribution + regional differences**.


## Task 2 — Sunburst: tipping behaviour by day and meal time

**What to build:** A sunburst chart using the built-in `tips` dataset showing how **total bill amount** is distributed across day → time → smoker status.

**Requirements:**
- Load tips with `px.data.tips()`
- Aggregate **total bill** (sum of `total_bill`) per group — not count
- Hierarchy: `path=['day', 'time', 'smoker']`
- Colour encodes smoker status with a CVD-safe blue/orange palette
- Grey out parent nodes (day and time level)
- Use `percent parent` for text labels
- Insight title describing where the most spending happens

> 💡 `tips.groupby(['day', 'time', 'smoker'])['total_bill'].sum().reset_index()`


In [3]:
# Task 2
# Load tips dataset
tips = px.data.tips()

# Aggregate total bill
tips_grouped = (
    tips.groupby(['day', 'time', 'smoker'])['total_bill']
    .sum()
    .reset_index()
)

# Find highest spending combination
top_group = tips_grouped.loc[
    tips_grouped['total_bill'].idxmax()
]

# Create sunburst chart
fig = px.sunburst(
    tips_grouped,
    path=['day', 'time', 'smoker'],
    values='total_bill',
    color='smoker',
    color_discrete_sequence=['#1f77b4', '#ff7f0e']
)

# Show percentages relative to parent
fig.update_traces(
    textinfo='label+percent parent'
)

# Add insight to title
fig.update_layout(
    title=f'Tipping Behaviour: Highest spending = {top_group["day"]} ({top_group["time"]})'
)

fig.show()


### Insights for Task 2 (Sunburst Chart — Tipping Behaviour)

1. **Dinner periods contribute more to total spending compared to lunch, indicating customers tend to spend higher amounts during evening meals.**

2. **The largest sunburst segments represent the day and time combinations with the highest restaurant spending, showing peak customer activity periods.**

3. **Differences between smoker and non-smoker groups suggest spending behaviour varies across customer types, though the impact differs by day and meal time.**

4. **The highest spending combination highlighted in the title identifies the customer group contributing most to total restaurant revenue.**

These insights explain **day-wise spending patterns, meal-time effects, smoker influence, and highest spending groups**, matching the chart objectives.


## Task 3 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country


In [4]:
# Task 3 — charts
# Filter low-carbon sources
low_carbon = df[df['Source_Type'] == 'Low-carbon']

# Aggregate total low-carbon generation by country
country_low = (
    low_carbon.groupby('Country')['TWh']
    .sum()
    .reset_index()
)

# Find leading country
top_country = country_low.loc[
    country_low['TWh'].idxmax(),
    'Country'
]

# ---------- Treemap ----------
country_low['Category'] = 'Low-carbon Energy'

fig1 = px.treemap(
    country_low,
    path=['Category', 'Country'],
    values='TWh',
    color='TWh',
    color_continuous_scale='Greens'
)

fig1.update_traces(textinfo='label+value')

fig1.update_layout(
    title='Low-carbon Energy Contribution by Country'
)

fig1.show()


# ---------- Horizontal Bar Chart ----------
country_low = country_low.sort_values(
    by='TWh',
    ascending=True
)

fig2 = px.bar(
    country_low,
    x='TWh',
    y='Country',
    orientation='h',
    color='TWh',
    color_continuous_scale='Greens',
    title=f'Leading Low-carbon Country: {top_country}'
)

fig2.update_layout(
    xaxis_title='Energy Generation (TWh)',
    yaxis_title='Country'
)

fig2.show()


# Insight
print(
    f"Insight: {top_country} produces the highest amount "
    "of low-carbon energy, indicating stronger adoption "
    "of cleaner energy sources."
)

Insight: France produces the highest amount of low-carbon energy, indicating stronger adoption of cleaner energy sources.


### Insights for Task 3 (Low-carbon Energy Contribution)

1. **Countries with larger treemap areas generate more low-carbon energy, indicating stronger adoption of cleaner and sustainable energy sources.**

2. **The horizontal bar chart highlights significant differences in low-carbon energy production between countries, showing uneven progress toward renewable energy transition.**

3. **The leading country in low-carbon energy generation demonstrates greater investment in sustainable energy infrastructure compared to others.**

4. **Countries with lower contributions may still rely more heavily on fossil fuels, suggesting opportunities to expand renewable and low-carbon energy adoption.**

These insights explain **country contributions, comparison across countries, leadership in clean energy, and sustainability trends**, matching the objectives of Task 3.
